In [1]:
# --- CELL 1: SETUP & DATA LOADING ---
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
import requests
import string

# Download Shakespeare's text directly (Project Gutenberg/Karpathy source)
# This fulfills the requirement to use a public domain dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
data = requests.get(url).text

print(f"Data Loaded. First 100 characters:\n{data[:100]}")

Data Loaded. First 100 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [2]:
# --- CELL 2: PREPROCESSING ---

def clean_text(text):
    """
    Cleans text by converting to lowercase and removing punctuation
    as per task instructions[cite: 18].
    """
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

# We limit to 100,000 characters to ensure training finishes within the 2-hour limit.
corpus = clean_text(data[:100000])

# Tokenize the text [cite: 19]
tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])
total_words = len(tokenizer.word_index) + 1

# Create input sequences and labels
input_sequences = []
token_list = tokenizer.texts_to_sequences([corpus])[0]

# Create n-gram sequences (Context window of 50 words)
sequence_length = 50
for i in range(1, len(token_list)):
    n_gram_sequence = token_list[max(0, i-sequence_length):i+1]
    input_sequences.append(n_gram_sequence)

# Pad sequences [cite: 20]
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Split into Input (X) and Output (y)
X, y = input_sequences[:,:-1], input_sequences[:,-1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Preprocessing complete.")
print(f"Total words: {total_words}")
print(f"Input Shape: {X.shape}")

Preprocessing complete.
Total words: 3080
Input Shape: (17892, 50)


In [3]:
# --- CELL 3: MODEL DESIGN & TRAINING ---

# Build the LSTM Model [cite: 21, 22]
model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_sequence_len-1)) # Embedding Layer
model.add(LSTM(150)) # LSTM Layer
model.add(Dense(total_words, activation='softmax')) # Output Layer

# Compile Model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model.summary())

# Train Model with Early Stopping
early_stop = EarlyStopping(monitor='loss', patience=3, verbose=1)

# Training (approx 10-15 mins on GPU)
history = model.fit(X, y, epochs=50, verbose=1, validation_split=0.2, callbacks=[early_stop])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.0360 - loss: 6.9049 - val_accuracy: 0.0363 - val_loss: 6.5567
Epoch 2/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.0422 - loss: 6.2560 - val_accuracy: 0.0475 - val_loss: 6.6110
Epoch 3/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.0530 - loss: 6.0747 - val_accuracy: 0.0464 - val_loss: 6.6287
Epoch 4/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.0541 - loss: 5.9098 - val_accuracy: 0.0548 - val_loss: 6.6111
Epoch 5/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.0682 - loss: 5.6800 - val_accuracy: 0.0595 - val_loss: 6.6121
Epoch 6/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.0808 - loss: 5.4435 - val_accuracy: 0.0659 - val_loss: 6.6553
Epoch 7/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.0915 - loss: 5.2439 - val_accuracy: 0.0659 - val_loss: 6.7078
Epoch 8/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.1110 - loss: 4.9933 - val_

In [4]:
# --- CELL 4: TEXT GENERATION ---

def generate_text(seed_text, next_words, model, max_sequence_len):
    """
    Generates new text based on a seed sequence[cite: 32].
    """
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

        # Predict next token
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

# Test with different seeds
seeds = [
    "the king said to",
    "shall i compare thee",
    "to be or not to be"
]

print("--- GENERATED OUTPUTS ---")
for s in seeds:
    generated = generate_text(s, 20, model, max_sequence_len)
    print(f"Seed: '{s}'\nResult: {generated}\n")

--- GENERATED OUTPUTS ---
Seed: 'the king said to'
Result: the king said to the capitol and will you be gone if they are undone already menenius and i inform them of you and

Seed: 'shall i compare thee'
Result: shall i compare thee not in him to you thus aufidius is all his words i end they nourishd disobedience fed the ruin of

Seed: 'to be or not to be'
Result: to be or not to be consul coriolanus all coriolanus pray you may not thus my good report i have been my son i therein would

